<div style="background:linear-gradient(90deg,#0366d6,#28a745); color:white; padding:20px 32px; border-radius:8px; width:97%;">

# 🧱 Lesson 3 · 本节精华

</div>

<small>对应主课:[3_a_🔥_subagents.ipynb](./3_a_🔥_subagents.ipynb)</small>

## 🎯 一句话定位

> 把任务**派发给专职子 agent**,每个子 agent 在**自己独立的 context 窗口**里跑 —— 用"开新 context"对抗 4 类 context 病。
>
> 🔒 **本质 = context isolation**(语境隔离),是 multi-agent 架构的基石。

## 📚 你应该学到的 7 件事

| # | 概念 | 你应该记住的一句话 |
|---|---|---|
| 1 | 🛡️ **4 类 context 病** | clash / confusion / poisoning / dilution(详见下方表) |
| 2 | 🔒 **Context Isolation** | 给每个子任务**开新 context**,只塞当前任务描述 |
| 3 | 🎭 **`SubAgent` 双重身份** | 既是工具(主 agent 调用),又是 agent(自己跑 ReAct) |
| 4 | 📤 **`task` 工具** | 主 agent 通过它**派单**:`task(description, subagent_type)` |
| 5 | 🪒 **核心一行** | `state["messages"] = [{...description}]` —— **擦掉父历史** |
| 6 | 🚀 **并行子 agent** | 主 agent 一次发出多个 `task` → 并发执行,**`max_concurrent` 限流** |
| 7 | 📜 **Scaling Rules** | prompt 教 LLM 何时单 / 何时多 sub-agent(对比题用多个,单题用一个) |

## 🦠 4 类 "Context 病" + 对应解药

| 病名 | 症状 | 例子 |
|---|---|---|
| 💥 **Context Clash**(冲突)| 同一 context 里混着相互矛盾的目标 | 一个 agent 同时被要求"调研 OpenAI"和"调研 Anthropic",可能信息串台 |
| 😵 **Context Confusion**(混乱)| 相似信息互相干扰 | 多家 AI 公司的安全做法描述太像,LLM 难以归位 |
| ☠️ **Context Poisoning**(污染)| 错误信息污染后续推理 | 早期一个搜索结果有事实错误,后续所有推理都基于错误 |
| 🧊 **Context Dilution**(稀释)| 关键信息被噪音淹没 | 50 次工具调用后,原始问题被埋在最上方 |

**共同解药 = 开新 context 跑专职子 agent**。每个子 agent 只看一个任务,信息互不串台。

## 💎 `task` 工具的**核心一行**

整个 `task` 工具看起来复杂,但灵魂只有一行:

<pre style="background:#f6f8fa; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">def task(description, subagent_type, state, tool_call_id):
    sub_agent = agents[subagent_type]

    <span style="background:#fff3a3; padding:0 2px;">state["messages"] = [{"role": "user", "content": description}]</span>  # ← 🔥 灵魂

    result = sub_agent.invoke(state)

    return Command(update={
        "files":    result.get("files", {}),                                      # 文件改动合并回父
        "messages": [ToolMessage(result["messages"][-1].content, tool_call_id=...)],  # 子 agent 答案 → 父的 ToolMessage
    })
</code></pre>

**这一句做了什么?**:把 state 里的 messages **彻底擦掉**,只留一条 user message(任务描述)。子 agent 拿到这个"清空过的 state"启动 —— **看不到父的任何对话历史**。

**但 state 的其他字段(files、todos)保留**,所以子 agent 能继承父的文件系统、todo 列表等。

## 🎭 `SubAgent` 的**双重身份**

回看 `SubAgent` 的 TypedDict:

```python
class SubAgent(TypedDict):
    name: str                        # 唯一标识
    description: str                 # 派单时主 agent 看的描述
    prompt: str                      # 系统 prompt(指导子 agent 怎么干活)
    tools: NotRequired[list[str]]    # 可选:限定它能用哪些工具
```

**每个字段对应它的一个"身份"**:

| 身份 | 需要的字段 | 作用 |
|---|---|---|
| 🔧 **作为工具(给主 agent 看)** | `name` + `description` | 告诉主 agent **何时调用、能做什么**(填进 `task` 工具的 schema) |
| 🤖 **作为 agent(自己运行时)** | `prompt` + `tools` | 告诉子 agent **怎么干活、能用什么工具**(传给 `create_agent`) |

> 🔑 **设计精髓**:**同一个配置对象,服务两个不同的消费者**(主 agent 决策时 / 子 agent 执行时)—— 这种"统一配置 + 双视角呈现"是 multi-agent 框架的通用模式。

## 🚀 并行子 agent:**多任务一次性派出去**

主 agent 可以在**同一个 AIMessage** 里发出多个 `task` tool_call → ToolNode 并发执行多个 sub-agent:

```python
# 主 agent 决定并行研究 3 家公司的 AI safety
AIMessage(tool_calls=[
    {'name': 'task', 'args': {'description': 'OpenAI 的 AI safety 做法',     'subagent_type': 'research-agent'}},
    {'name': 'task', 'args': {'description': 'Anthropic 的 AI safety 做法',  'subagent_type': 'research-agent'}},
    {'name': 'task', 'args': {'description': 'DeepMind 的 AI safety 做法',   'subagent_type': 'research-agent'}},
])
```

**3 个子 agent 同时跑**,每个用自己的隔离 context,各自把成果文件提交回来,父 agent 汇总。

**这里 `file_reducer` 的合并能力是必需的** —— 3 个并行子 agent 各自写不同的 `findings_*.md`,**没有合并 reducer 会互相覆盖**(这就是 Lesson 2 为什么要定义 `file_reducer`)。

## 🌍 真实产品对应 + 一句话

| 你学的 | 现实里的实现 |
|---|---|
| 🔒 Context isolation | **Anthropic Multi-Agent Research System**,本节原型来源 |
| 🎭 SubAgent 双重身份 | **Claude Code 的 general-purpose agent**(被 Task tool 派遣) |
| 🚀 并行 sub-agents | **Devin / Cursor Composer**:supervisor 派多个 worker 并发 |
| 📜 Scaling Rules | LangGraph 1.0 推荐的 multi-agent prompt 范式 |

## ✨ 一句话带走

<div style="background:#e7f7e9; padding:10px 24px; border-left:4px solid #28a745; border-radius:4px; width:97%;">

> 🔑 **子 agent = 开新 context 的"用完即弃"工人**。主 agent 是 supervisor,只看摘要;每个 sub-agent 在干净 context 里专心干一件事,然后把成果**摘要 + 文件**交回父。
>
> **`state["messages"] = [{...description}]` 这一行 = 隔离的全部秘诀**。剩下的都是工程细节。

</div>